# 아두이노 RGB 센서 기반 과일 분류 모델

## 프로젝트 개요
- **목표**: Fruits-360 데이터셋으로 EfficientNet-B3 학습 후 RGB 데이터로 과일 판별
- **입력**: 아두이노 RGB 센서 3개 (위, 아래, 앞) → 9개 RGB 값
- **출력**: 과일 종류 분류
- **예상 정확도**: 80-90%

## 실행 순서
1. Fruits-360 데이터 로드
2. EfficientNet-B3 전이학습 모델 구축 및 학습

## 1. Fruits-360 데이터 로드

In [ ]:
# 필요한 라이브러리 임포트
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import timm
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tqdm

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 디바이스: {device}")

# 랜덤 시드 고정
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Fruits-360 데이터셋 경로 설정
dataset_base_path = '../Fruit-Images-Dataset'

# 데이터 경로 확인
train_path = os.path.join(dataset_base_path, 'Training')
test_path = os.path.join(dataset_base_path, 'Test')

if not os.path.exists(train_path):
    print(f"⚠️ Training 데이터를 찾을 수 없습니다: {train_path}")
    print("Fruits-360 데이터를 다운로드해주세요!")
else:
    # 클래스 정보 확인
    classes = os.listdir(train_path)
    num_classes = len(classes)
    
    print(f"✅ Training 데이터셋 경로: {train_path}")
    print(f"클래스 수: {num_classes}")
    print(f"처음 10개 클래스: {classes[:10]}")

In [ ]:
# 데이터 전처리 설정
image_size = 224  # EfficientNet 입력 크기

transform_train = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

transform_test = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 데이터셋 로드
print("📦 데이터셋 로드 중...")
train_dataset = ImageFolder(root=train_path, transform=transform_train)
test_dataset = ImageFolder(root=test_path, transform=transform_test)

# 클래스 맵핑 저장
class_to_idx = train_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

print(f"✅ Training 데이터: {len(train_dataset)}개")
print(f"✅ Test 데이터: {len(test_dataset)}개")
print(f"클래스 맵핑 저장 완료")

In [ ]:
# 데이터 로더 생성
batch_size = 32
num_workers = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

print(f"✅ DataLoader 생성 완료")
print(f"Training 배치: {len(train_loader)}")
print(f"Test 배치: {len(test_loader)}")

In [ ]:
# 샘플 이미지 시각화
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.ravel()

for i in range(6):
    # 데이터로더에서 배치 하나 가져오기
    images, labels = next(iter(train_loader))
    
    # 첫 번째 이미지
    img = images[0]
    label = labels[0]
    
    # 정규화 해제 (시각화용)
    img = img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    img = img.permute(1, 2, 0).numpy().clip(0, 1)
    
    class_name = idx_to_class[label.item()]
    
    axes[i].imshow(img)
    axes[i].set_title(class_name, fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('sample_fruits.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ 샘플 이미지 저장: sample_fruits.png")

## 2. EfficientNet-B3 전이학습 모델 구축 및 학습

In [ ]:
# EfficientNet-B3 모델 로드 (ImageNet 사전학습)
print("📥 EfficientNet-B3 모델 다운로드 중...")
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=num_classes)
model = model.to(device)

print(f"✅ 모델 로드 완료")
print(f"총 클래스: {num_classes}")

# 모델 구조 확인
print("\n모델 정보:")
print(f"  - 입력: 3채널 {image_size}x{image_size} 이미지")
print(f"  - 출력: {num_classes}개 클래스")
print(f"  - 모델 파라미터: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

In [ ]:
# 학습 설정
learning_rate = 0.001
num_epochs = 10
weight_decay = 1e-4

# Loss와 Optimizer 정의
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

print("🎯 학습 설정:")
print(f"  - Epochs: {num_epochs}")
print(f"  - Batch Size: {batch_size}")
print(f"  - Learning Rate: {learning_rate}")
print(f"  - Optimizer: AdamW")
print(f"  - Scheduler: CosineAnnealingLR")

In [ ]:
# 학습 함수
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    progress_bar = tqdm.tqdm(train_loader, desc='Training')
    
    for images, labels in progress_bar:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 통계
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        progress_bar.set_postfix({
            'loss': total_loss / (progress_bar.n + 1),
            'acc': 100 * correct / total
        })
    
    return total_loss / len(train_loader), 100 * correct / total


def evaluate(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        progress_bar = tqdm.tqdm(test_loader, desc='Evaluating')
        
        for images, labels in progress_bar:
            images = images.to(device)
            labels = labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # 통계
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
            progress_bar.set_postfix({
                'loss': total_loss / (progress_bar.n + 1),
                'acc': 100 * correct / total
            })
    
    return total_loss / len(test_loader), 100 * correct / total

print("✅ 학습 함수 정의 완료")

In [ ]:
# 전체 학습 루프
history = {
    'train_loss': [],
    'train_acc': [],
    'test_loss': [],
    'test_acc': []
}

best_acc = 0
best_model_path = 'efficientnet_b3_fruits.pth'

print("🚀 학습 시작!\n")

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"{'='*50}")
    
    # 학습
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # 평가
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    # 스케줄러 업데이트
    scheduler.step()
    
    # 히스토리 저장
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    # 결과 출력
    print(f"\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc:.2f}%")
    
    # 최고 성능 모델 저장
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"✅ 최고 성능 모델 저장! (Acc: {test_acc:.2f}%)")
    
    # 현재 학습률 출력
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Learning Rate: {current_lr:.6f}")

print(f"\n{'='*50}")
print(f"🎉 학습 완료!")
print(f"최고 테스트 정확도: {best_acc:.2f}%")
print(f"{'='*50}")

In [ ]:
# 학습 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss 그래프
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['test_loss'], label='Test Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Progress')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy 그래프
axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o')
axes[1].plot(history['test_acc'], label='Test Accuracy', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Progress')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ 학습 결과 저장: training_history.png")

In [ ]:
# 최고 성능 모델 로드 및 저장
print("📥 최고 성능 모델 로드 중...")
model.load_state_dict(torch.load(best_model_path, map_location=device))

# 모델 정보 저장
model_info = {
    'model_name': 'efficientnet_b3',
    'num_classes': num_classes,
    'class_to_idx': class_to_idx,
    'idx_to_class': idx_to_class,
    'image_size': image_size,
    'best_acc': best_acc
}

import json
with open('model_info.json', 'w') as f:
    # idx_to_class의 키를 문자열로 변환 (JSON 호환)
    info_to_save = model_info.copy()
    info_to_save['idx_to_class'] = {str(k): v for k, v in idx_to_class.items()}
    json.dump(info_to_save, f, indent=4)

print(f"✅ 모델 저장 완료")
print(f"  - 모델 파일: {best_model_path}")
print(f"  - 모델 정보: model_info.json")
print(f"\n📊 최종 성능:")
print(f"  - 최고 테스트 정확도: {best_acc:.2f}%")
print(f"  - 모델 파라미터: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"  - 클래스 수: {num_classes}")